Saved outputs are retained from the original experiments; they have not been regenerated with the cleaned implementation. Set data and checkpoint paths below before running.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "count_fm").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

In [1]:
# Select a GPU externally with CUDA_VISIBLE_DEVICES if needed.
import os

In [2]:
import torch
print("cuda?", torch.cuda.is_available(), "n_dev", torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda? True n_dev 1
NVIDIA GeForce RTX 4090


In [3]:
import os, sys, math, time, random
import numpy as np
import pandas as pd

SEED = 0
def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)

# Paths (match your existing notebook default)
DG_H5AD = "read_data/datasets/dentate_gyrus/DentateGyrus.h5ad"


# Train/test split
TEST_FRAC = 0.2

# Comparison budget
N_GEN = None  # if None, generate same number as X_test
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Featurizer for metrics (shared across models)
TARGET_SUM = 1e4
G_KEEP = 256   # select top-variance genes before PCA
PCA_DIM = 50   # set None to disable PCA (not recommended)

print("Config loaded.")

Config loaded.


# 1. Load Data

In [4]:
import scanpy as sc
import scipy.sparse as sp

adata = sc.read_h5ad(DG_H5AD)

# Prefer your layer if present
if "matrix" in adata.layers:
    X = adata.layers["matrix"]
else:
    X = adata.X

if sp.issparse(X):
    X = X.toarray()

# Ensure integer-like counts
X = np.asarray(X)
if not np.issubdtype(X.dtype, np.integer):
    # some h5ad store counts as float
    X = np.rint(X).astype(np.int64)

N, d = X.shape
print("Loaded X:", X.shape, X.dtype, "min/max:", X.min(), X.max())

Loaded X: (2930, 13913) int64 min/max: 0 1718


## train-test split

In [5]:
idx = np.arange(N)
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

n_test = int(round(TEST_FRAC * N))
test_idx = idx[:n_test]
train_idx = idx[n_test:]

X_train = X[train_idx]
X_test  = X[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (2344, 13913) Test: (586, 13913)


## featurizer

In [6]:
from count_fm.utils.scrna_metrics import log1p_libnorm_np, Featurizer

from sklearn.decomposition import PCA



feat = Featurizer(target_sum=TARGET_SUM, g_keep=G_KEEP, pca_dim=PCA_DIM, seed=SEED).fit(X_train)
Z_test = feat.transform(X_test)
print("Feature dim:", Z_test.shape)

Feature dim: (586, 50)


# 2. metrics

In [7]:
from count_fm.utils.scrna_metrics import w2_gaussian, mmd2_rbf_unbiased, _median_heuristic_sigma2

# 3. Models

In [8]:
from count_fm.experiments.base import UncondModel

## 3.1 count-FM

In [9]:
import count_fm as countfm
import count_fm.models as models
from count_fm.experiments.scrna_countfm import CountFMModel, sample_euler_batched

## 3.2 scVI

In [10]:
from count_fm.baselines.scrna.scvi import SCVIModel

## 3.3 scDiffusion

In [11]:
# !pip -q install blobfile mpi4py
# !git clone https://github.com/EperLuo/scDiffusion scDiffusion

In [12]:
from count_fm.baselines.scrna.scdiffusion import ScDiffusionModel

## 3.4 CFGen

In [13]:
# %%bash
# set -e

# # If CFGen exists and is a git repo, update it. Otherwise, remove and re-clone.
# if [ -d CFGen/.git ]; then
#   git -C CFGen fetch --all
#   git -C CFGen checkout main
#   git -C CFGen pull
# else
#   rm -rf CFGen
#   git clone https://github.com/theislab/CFGen CFGen
# fi

# # Patch packaging: setup.py expects README.rst but repo has README.md
# if [ -f CFGen/README.md ] && [ ! -f CFGen/README.rst ]; then
#   cp CFGen/README.md CFGen/README.rst
# fi

# pip install -e CFGen

# python -c "import cfgen; print('cfgen import OK:', cfgen.__file__)"

In [14]:
# !git clone https://github.com/theislab/CFGen CFGen
# !pip -q install -e CFGen

In [15]:
from count_fm.baselines.scrna.cfgen import _torch_load_trusted, CFGenModel

## 3.5 scLDM

In [16]:
from count_fm.baselines.scrna.scldm import SCLDMModel

## 3.6 DCM

In [17]:
# !git clone https://github.com/sanjukta7/aivc-dcm aivc-dcm

In [18]:
from count_fm.baselines.scrna.dcm import DCMModel

## 3.7 save and load

In [19]:
from count_fm.experiments.scrna_countfm import save_countfm, load_countfm, init_or_load_countfm_ot

In [20]:
from count_fm.baselines.scrna.scvi import save_scvi, load_scvi

In [21]:
from count_fm.baselines.scrna.scdiffusion import save_scdiffusion_bundle, load_scdiffusion_bundle

In [22]:
from count_fm.baselines.scrna.cfgen import _jsonify, _torch_load_weights, _infer_encoder_dims_and_d_from_sd, _remap_keys_to_match_target, save_cfgen_bundle, load_cfgen_bundle

In [23]:
from count_fm.baselines.scrna.scldm import save_scldm_bundle, load_scldm_bundle

In [24]:
from count_fm.baselines.scrna.dcm import save_dcm_bundle, load_dcm_bundle

# 4. Training

## 4.1 Instantiate

In [25]:
# Instantiate
countfm_model = CountFMModel(
    device=DEVICE,
    num_epochs=500,
    batch_size=32,
    eps_t=1e-4,
    eps_log=1e-8,
    loss_mode="poisson",
    margin=2,
    C_max=None,

    # match 3_scRNA.ipynb behavior
    x0_mode_train="poisson",
    lam0_train=1.0,

    # OT coupling option (set "none" first; later try "sym_poisson"/"L2"/"L1")
    ot_cost="none",
    # ot_cost="sym_poisson",
    
    # transformer backbone (same family as 3_scRNA.ipynb)
    d_model=256,
    depth=8,
    n_heads=8,
    mlp_ratio=4.0,
    dropout=0.0,
    chunk_size=128,
    log_cap=50.0,
)

In [26]:
scvi_model = SCVIModel(
    device=DEVICE,
    n_latent=30,
    n_layers=3,
    n_hidden=256,
    gene_likelihood="zinb",
    max_epochs=800,
)

In [27]:
scdiff_model = ScDiffusionModel(
    scd_root="scDiffusion",
    workdir="saved_models/scdiffusion_dg_v1",
    scimilarity_dir=None,
    device=DEVICE,
    ae_steps=150000, # 150000
    diff_steps=600000, # 600000
    diff_save_interval=100000, # 100000
    batch_size=128,
)

In [28]:
cfgen_model = CFGenModel(
    device=DEVICE,
    workdir="saved_models/cfgen_dg_v1",
    seed=SEED,

    # unconditional setup
    covariate_key="dummy",
    fm_conditional=False,

    # you can cut epochs for quick tests
    encoder_max_epochs=500,
    fm_max_epochs=3000,
)

In [29]:
DIR_SCLDM = "saved_models/scldm_dg_v1"

SCLDM_PY   = os.environ.get("SCLDM_PYTHON", sys.executable)
SCLDM_REPO = os.environ.get("SCLDM_ROOT", "external/scLDM")

scldm_model = SCLDMModel(
    scldm_root=SCLDM_REPO ,
    py_exec=SCLDM_PY,
    workdir=DIR_SCLDM,
    seed=SEED,
    device="cuda",
    vae_epochs=100, # 100
    ldm_epochs=100, # 100
    batch_size=8,
    test_batch_size=16,
    num_workers=0,
    timesteps=100,
    guidance_weight=0.0,
)

In [30]:
DIR_DCM = "saved_models/dcm_dg_bundle_v1"

# Use a python>=3.10 interpreter if your notebook kernel is older.
# If your current kernel is already >=3.10 and has deps, you can set DCM_PY = None.
DCM_PY   = None
DCM_REPO = "aivc-dcm"

dcm_model = DCMModel(
    dcm_root=DCM_REPO,
    py_exec=DCM_PY,
    workdir="saved_models/dcm_dg_workdir_v1",
    seed=SEED,
    device=DEVICE,

    # defaults match rnaseq_small.yaml (small transformer)
    hidden_dim=256,
    num_layers=8,
    num_heads=8,
    dropout=0.1,

    batch_size=32,
    num_epochs=50,
    learning_rate=5e-4,
    weight_decay=0.0,
    mask_ratio=0.15,
    val_fraction=0.1,
    num_workers=0,
    save_interval=2,

    default_num_steps=100,
    default_temperature=1.0,

    apply_mask_token_fix=True,
)

## 4.2 fitting

### initial training

In [31]:
# !rm -rf saved_models/scldm_dg_v1/experiments/checkpoints/vae_dg_custom
# !rm -rf saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom

In [32]:
# # count-FM (fit already accepts lr/wd/epochs)
# t0 = time.time()
# countfm_model.fit(X_train, lr=5e-4,
#                   weight_decay=1e-4, num_epochs=500, resume=True)
# print("count-FM trained in", round(time.time() - t0, 1), "sec")


# # scVI (fit already accepts lr/wd/max_epochs)
# t0 = time.time()
# scvi_model.fit(X_train, lr=5e-4,
#                weight_decay=1e-4, max_epochs=500, resume=True)
# print("scVI trained in", round(time.time() - t0, 1), "sec")


# # scDiffusion (fit accepts lr/wd, but steps/batch live on the object)
# scdiff_model.ae_steps = 150000
# scdiff_model.diff_steps = 600000
# scdiff_model.diff_save_interval = 100000
# scdiff_model.batch_size = 128

# t0 = time.time()
# scdiff_model.fit(X_train, lr=5e-4,
#                  weight_decay=1e-4, resume=True)
# print("scDiffusion trained in", round(time.time() - t0, 1), "sec")


# # CFGen (fit doesn’t expose optimizer/epochs, but the object has them)
# cfgen_model.encoder_max_epochs = 300
# cfgen_model.fm_max_epochs = 1500
# cfgen_model.encoder_lr = 1e-3
# cfgen_model.encoder_wd = 1e-5
# cfgen_model.fm_lr = 1e-4
# cfgen_model.fm_wd = 1e-6

# t0 = time.time()
# cfgen_model.fit(X_train, var_names=adata.var_names, resume=True)
# print("CFGen trained in", round(time.time() - t0, 1), "sec")


# # scLDM (fit doesn’t expose lr; you can still change budgets + sampler knobs)
# scldm_model.vae_epochs = 100
# scldm_model.ldm_epochs = 100
# scldm_model.batch_size = 8
# scldm_model.test_batch_size = 128
# scldm_model.timesteps = 100
# scldm_model.guidance_weight = 0.0

# t0 = time.time()
# scldm_model.fit(X_train, X_test, var_names=adata.var_names, resume=True)
# print("scLDM trained in", round(time.time() - t0, 1), "sec")


# # DCM (your DCMModel.fit already supports overriding these via kwargs)
# t0 = time.time()
# dcm_model.fit(
#     X_train,
#     var_names=adata.var_names,
#     resume=True,
#     batch_size=8,
#     num_epochs=50,
#     learning_rate=1e-4,
#     weight_decay=1e-2,
# )
# print("DCM trained in", round(time.time() - t0, 1), "sec")

### continue training

In [33]:
# DIR_COUNTFM = "saved_models/countfm_dg_v1"
DIR_COUNTFM = "saved_models/backup/countfm_dg_v2"
DIR_COUNTFM_OT = "saved_models/backup/countfm_dg_v2_OT"

DIR_SCVI    = "saved_models/scvi_dg_v1"
DIR_SCD     = "saved_models/scdiffusion_dg_bundle_v1"
DIR_CFGEN   = "saved_models/cfgen_dg_bundle_v1"
DIR_SCLDM   = "saved_models/scldm_dg_v1"
DIR_DCM     = "saved_models/dcm_dg_bundle_v1"

In [34]:
countfm_model = load_countfm(DIR_COUNTFM, device=DEVICE)
countfm_ot_model = init_or_load_countfm_ot(
    DIR_COUNTFM_OT,
    DIR_COUNTFM,
    device=DEVICE,
    ot_cost="sym_poisson",
)

try:
    scvi_model = load_scvi(DIR_SCVI, device=DEVICE)
except RuntimeError:
    scvi_model = load_scvi(DIR_SCVI, device=DEVICE, X_ref_counts=X_train)

scdiff_model = load_scdiffusion_bundle(DIR_SCD, device=DEVICE)
cfgen_model = load_cfgen_bundle(DIR_CFGEN, device=DEVICE)
scldm_model = load_scldm_bundle(DIR_SCLDM)
dcm_model = load_dcm_bundle(DIR_DCM)

Loaded count-FM from saved_models/backup/countfm_dg_v2
Loaded count-FM from saved_models/backup/countfm_dg_v2_OT
Loaded count-FM-OT from saved_models/backup/countfm_dg_v2_OT
INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  
INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Loaded scVI from saved_models/scvi_dg_v1
Loaded scDiffusion bundle: saved_models/scdiffusion_dg_bundle_v1
Loaded CFGen bundle (weights-only): saved_models/cfgen_dg_bundle_v1 | zdim = 50 | missing = 29 | unexpected = 26
Loaded scLDM bundle: saved_models/scldm_dg_v1


/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'encoder_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder_model'])`.
/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'denoising_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['denoising_model'])`.


In [35]:
SCVI_EPOCHS      = 800

SCD_AE_STEPS     = 150000
SCD_DIFF_STEPS   = 600000
SCD_SAVE_INT     = 100000
SCD_BATCH        = 128

CFGEN_ENC_EPOCHS = 500
CFGEN_FM_EPOCHS  = 3000

SCLDM_VAE_EPOCHS = 200
SCLDM_LDM_EPOCHS = 200
SCLDM_BATCH      = 8
SCLDM_TEST_BATCH = 128

DCM_EPOCHS       = 100
DCM_BATCH        = 8
DCM_LR           = 5e-4
DCM_WD           = 0.0


COUNTFM_OT_EPOCHS = 500
COUNTFM_OT_COST = "sym_poisson"
CONTINUE_COUNTFM_OT = False

countfm_ot_model.ot_cost = COUNTFM_OT_COST

if CONTINUE_COUNTFM_OT:
    t0 = time.time()
    countfm_ot_model.fit(
        X_train,
        lr=1e-4,
        weight_decay=0.0,
        num_epochs=COUNTFM_OT_EPOCHS,
        resume=True,
    )
    countfm_ot_model._needs_ot_training = False
    print("count-FM-OT refine:", round(time.time() - t0, 1), "sec")
else:
    print("Skip count-FM-OT continuation training.")



# # count-FM (fit already accepts lr/wd/epochs)
# t0 = time.time()
# countfm_model.fit(X_train, lr=1e-4,
#                   weight_decay=0.0, num_epochs=50, resume=True)
# print("count-FM trained in", round(time.time() - t0, 1), "sec")



# # scVI
# t0 = time.time()
# scvi_model.fit(X_train, lr=2e-4, weight_decay=0.0,
#                max_epochs=SCVI_EPOCHS, resume=True)
# print("scVI refine:", round(time.time()-t0, 1), "sec")


# # scDiffusion (step-based, just raise targets then resume)
# scdiff_model.ae_steps = SCD_AE_STEPS
# scdiff_model.diff_steps = SCD_DIFF_STEPS
# scdiff_model.diff_save_interval = SCD_SAVE_INT
# scdiff_model.batch_size = SCD_BATCH

# t0 = time.time()
# scdiff_model.fit(X_train, lr=5e-4, weight_decay=0.0, resume=True)
# print("scDiff refine:", round(time.time()-t0, 1), "sec")


# # CFGen (Lightning resumes from last.ckpt; raise max_epochs)
# cfgen_model.encoder_max_epochs = CFGEN_ENC_EPOCHS
# cfgen_model.fm_max_epochs = CFGEN_FM_EPOCHS

# t0 = time.time()
# cfgen_model.fit(X_train, var_names=adata.var_names, resume=True)
# print("CFGen refine:", round(time.time()-t0, 1), "sec")


# # scLDM (Lightning resumes from last.ckpt; raise num_epochs)
# scldm_model.vae_epochs = SCLDM_VAE_EPOCHS
# scldm_model.ldm_epochs = SCLDM_LDM_EPOCHS
# scldm_model.batch_size = SCLDM_BATCH
# scldm_model.test_batch_size = SCLDM_TEST_BATCH



# import os, json, subprocess, re

# def _ckpt_epoch_via_py(py_exec, ckpt_path):
#     if not os.path.exists(ckpt_path):
#         return None
#     code = r"""
# import sys, torch
# p = sys.argv[1]
# ckpt = torch.load(p, map_location="cpu", weights_only=False)
# ep = ckpt.get("epoch", None)
# gs = ckpt.get("global_step", None)
# print(f"{ep} {gs}")
# """
#     r = subprocess.run([py_exec, "-c", code, ckpt_path], capture_output=True, text=True)
#     if r.returncode != 0:
#         print("ckpt read failed:", r.stderr[:300])
#         return None
#     m = re.findall(r"(-?\d+|None)", r.stdout.strip())
#     ep = None if (len(m) < 1 or m[0] == "None") else int(m[0])
#     return ep

# vae_last = os.path.join(scldm_model.workdir, "experiments", "checkpoints", "vae_dg_custom", "last.ckpt")
# ldm_last = os.path.join(scldm_model.workdir, "experiments", "checkpoints", "ldm_dg_custom", "last.ckpt")

# vae_ep = _ckpt_epoch_via_py(scldm_model.py_exec, vae_last)
# ldm_ep = _ckpt_epoch_via_py(scldm_model.py_exec, ldm_last)

# print("scLDM current epochs:", "VAE =", vae_ep, "| LDM =", ldm_ep)

# # continue +50 epochs from current
# if vae_ep is not None:
#     scldm_model.vae_epochs = max(int(scldm_model.vae_epochs), int(vae_ep) + 50)
# if ldm_ep is not None:
#     scldm_model.ldm_epochs = max(int(scldm_model.ldm_epochs), int(ldm_ep) + 50)

# print("scLDM new targets:", "VAE =", scldm_model.vae_epochs, "| LDM =", scldm_model.ldm_epochs)


# t0 = time.time()
# scldm_model.fit(X_train, X_test, var_names=adata.var_names, resume=True)
# print("scLDM refine:", round(time.time()-t0, 1), "sec")


# # DCM (script auto-resumes; raise num_epochs)
# t0 = time.time()
# dcm_model.fit(X_train, var_names=adata.var_names, resume=True,
#               batch_size=DCM_BATCH, num_epochs=DCM_EPOCHS,
#               learning_rate=DCM_LR, weight_decay=DCM_WD)
# print("DCM refine:", round(time.time()-t0, 1), "sec")

Skip count-FM-OT continuation training.


In [36]:
# # ===== fast eval setup (cheap, for chunk monitoring only) =====
# EVAL_N = 300          # smaller = faster
# EVAL_G_KEEP = 512     # smaller gene subset for eval speed
# EVAL_PCA_DIM = 50     # keep small
# EVAL_SEED = 123

# feat_eval = Featurizer(
#     target_sum=TARGET_SUM,
#     g_keep=EVAL_G_KEEP,
#     pca_dim=EVAL_PCA_DIM,
#     seed=SEED,
# )
# # fit on TRAIN (or VAL) once
# feat_eval.fit(X_train)

In [37]:
# def _set_all_seeds(s):
#     s = int(s)
#     random.seed(s)
#     np.random.seed(s)
#     if torch is not None:
#         torch.manual_seed(s)
#         if torch.cuda.is_available():
#             torch.cuda.manual_seed_all(s)

# def _eval_one_chunk_fast(model, Zref, sigma2, n_samples, seed):
#     _set_all_seeds(seed)
#     if hasattr(model, "seed"):
#         model.seed = int(seed)

#     Xg = model.sample(
#         n_samples=int(n_samples),
#         n_step=COUNT_FM_NSTEP,
#         batch_eval=COUNT_FM_BATCH_EVAL,
#         x0_mode="poisson",
#         lam0=COUNT_FM_LAM0,
#     )
#     Zg = feat_eval.transform(Xg)  # fast featurizer

#     w2 = float(w2_gaussian(Zref, Zg))
#     mmd = float(mmd2_rbf_unbiased(Zref, Zg, sigma2=sigma2)[0])
#     return w2, mmd

In [38]:
# # make a validation split from training (do this once)
# VAL_FRAC = 0.2
# rng = np.random.default_rng(SEED)
# idx = rng.permutation(X_train.shape[0])
# n_val = int(round(VAL_FRAC * len(idx)))
# val_idx = idx[:n_val]
# tr_idx  = idx[n_val:]
# X_tr = X_train[tr_idx]
# X_val = X_train[val_idx]
# print("train/val:", X_tr.shape, X_val.shape)

# # ===== fast refs for early stopping =====
# N_EVAL = 400
# Xval_ref = X_val[:N_EVAL]
# Xte_ref  = X_test[:N_EVAL]

# Zval_ref = feat_eval.transform(Xval_ref)
# Zte_ref  = feat_eval.transform(Xte_ref)

# sigma2_val = _median_heuristic_sigma2(Zval_ref, max_points=1000, seed=SEED)
# sigma2_te  = _median_heuristic_sigma2(Zte_ref,  max_points=1000, seed=SEED)
# sigma2_val = max(sigma2_val, 1e-6)
# sigma2_te  = max(sigma2_te, 1e-6)

# print("feat_eval dim:", Zval_ref.shape[1], "| N_EVAL:", N_EVAL, "| EVAL_G_KEEP:", EVAL_G_KEEP)

In [39]:
# TOTAL = 2000
# CHUNK = 100
# PATIENCE = 2
# EVAL_SEED = 123
# N_EVAL = 400
# BEST_DIR = "saved_models/countfm_best_val"
# os.makedirs(BEST_DIR, exist_ok=True)

# best = {"mmd": float("inf"), "step": None}
# bad = 0

# for t in range(0, TOTAL, CHUNK):
#     countfm_model.fit(X_tr, lr=5e-4, weight_decay=0.0, num_epochs=CHUNK, resume=True)

#     # fixed-seed eval on VAL
#     w2_val, mmd_val = _eval_one_chunk_fast(countfm_model, Zval_ref, sigma2_val, n_samples=N_EVAL, seed=EVAL_SEED)
#     print(f"after {t+CHUNK:4d} | VAL W2 {w2_val:.3f} MMD {mmd_val:.4f}")

#     if mmd_val < best["mmd"]:
#         best.update({"mmd": mmd_val, "step": t+CHUNK})
#         bad = 0
#         save_countfm(countfm_model, BEST_DIR)
#         print("  [best] saved @", best["step"], "val_mmd", best["mmd"])
#     else:
#         bad += 1
#         if bad >= PATIENCE:
#             print("Early stop. Best step =", best["step"], "best val_mmd =", best["mmd"])
#             break

# # final evaluation on TEST using the best model
# countfm_best = load_countfm(BEST_DIR, device=DEVICE)
# w2_te, mmd_te = _eval_one_chunk_fast(countfm_best, Zte_ref, sigma2_te, n_samples=N_EVAL, seed=EVAL_SEED)
# print("BEST checkpoint TEST:", "W2", w2_te, "MMD", mmd_te)

In [40]:
# DIR_COUNTFM = "saved_models/backup/countfm_dg_v2"
# DIR_COUNTFM_OT = "saved_models/backup/countfm_dg_v2_OT"

In [41]:
TRAIN_COUNTFM_OT_V2 = True
DIR_COUNTFM_OT_V2 = "saved_models/backup/countfm_dg_v3_OT_b256_v3"

In [42]:
COUNTFM_OT_V2_BATCH = 256
COUNTFM_OT_V2_OTPAIR = 128
COUNTFM_OT_V2_EPOCHS = 500
COUNTFM_OT_V2_LR = 1e-4
COUNTFM_OT_V2_WD = 0.0

In [43]:
DIR_COUNTFM_OT

'saved_models/backup/countfm_dg_v2_OT'

In [44]:
# Train a new OT-v2 model by continuing from the old OT checkpoint,
# but with a larger training/OT batch. Save to a new directory.

countfm_ot_v2_model = load_countfm(DIR_COUNTFM, device=DEVICE)
# countfm_ot_v2_model = load_countfm(DIR_COUNTFM_OT_V2, device=DEVICE)

countfm_ot_v2_model.ot_cost = "sym_poisson"

# optimizer batch and OT candidate batch are now decoupled
countfm_ot_v2_model.batch_size = COUNTFM_OT_V2_BATCH
countfm_ot_v2_model.ot_pair_batch_size = COUNTFM_OT_V2_OTPAIR
countfm_ot_v2_model.num_epochs = COUNTFM_OT_V2_EPOCHS

if TRAIN_COUNTFM_OT_V2:
    t0 = time.time()

    countfm_ot_v2_model.fit(
        X_train,
        lr=COUNTFM_OT_V2_LR,
        weight_decay=COUNTFM_OT_V2_WD,
        num_epochs=COUNTFM_OT_V2_EPOCHS,
        resume=True,
    )

    save_countfm(countfm_ot_v2_model, DIR_COUNTFM_OT_V2)

    print("count-FM-OT v2 refine:",
          round(time.time() - t0, 1), "sec")
    print("Saved to:", DIR_COUNTFM_OT_V2)
else:
    print("Skip count-FM-OT v2 training.")
    print("When ready, set TRAIN_COUNTFM_OT_V2 = True")

Loaded count-FM from saved_models/backup/countfm_dg_v2


  0%|          | 0/500 [00:00<?, ?it/s]

/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/torch/utils/_device.py:103: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  return func(*args, **kwargs)


[CountFM][epoch 0] loss=-2596.747803
[CountFM][epoch 100] loss=-2663.887451
[CountFM][epoch 200] loss=-2197.019043
[CountFM][epoch 300] loss=-2986.969238
[CountFM][epoch 400] loss=-3383.923096
count-FM: resume=True lr=0.0001 wd=0.0 epochs=500 batch=256 ot_pair_batch=128
Saved count-FM to saved_models/backup/countfm_dg_v3_OT_b256_v3
count-FM-OT v2 refine: 384.3 sec
Saved to: saved_models/backup/countfm_dg_v3_OT_b256_v3


## 4.3 Save and Load

In [45]:
DIR_COUNTFM    = "saved_models/backup/countfm_dg_v2"
DIR_COUNTFM_OT = "saved_models/backup/countfm_dg_v2_OT"
DIR_SCVI    = "saved_models/scvi_dg_v1"
DIR_SCD     = "saved_models/scdiffusion_dg_bundle_v1"
DIR_CFGEN   = "saved_models/cfgen_dg_bundle_v1"
DIR_SCLDM   = "saved_models/scldm_dg_v1"
DIR_DCM     = "saved_models/dcm_dg_bundle_v1"

In [46]:
# if not getattr(countfm_ot_model, "_needs_ot_training", False):
#     save_countfm(countfm_ot_model, DIR_COUNTFM_OT)
# else:
#     print("count-FM-OT not saved, it is only initialized from non-OT weights.")

# save_scvi(scvi_model, DIR_SCVI)
# save_scdiffusion_bundle(scdiff_model, DIR_SCD)
# save_cfgen_bundle(cfgen_model, DIR_CFGEN)
# save_scldm_bundle(scldm_model, DIR_SCLDM)
# save_dcm_bundle(dcm_model, DIR_DCM)

In [47]:
# countfm_model = load_countfm(DIR_COUNTFM, device=DEVICE)

In [48]:
# try:
#     scvi_model = load_scvi(DIR_SCVI, device=DEVICE)
# except RuntimeError:
#     scvi_model = load_scvi(DIR_SCVI, device=DEVICE, X_ref_counts=X_train)

In [49]:
# scdiff_model = load_scdiffusion_bundle(DIR_SCD, device=DEVICE)

In [50]:
# cfgen_model = load_cfgen_bundle(DIR_CFGEN, device=DEVICE)

In [51]:
# scldm_model = load_scldm_bundle(DIR_SCLDM)

In [52]:
# dcm_model = load_dcm_bundle(DIR_DCM)

# 5. Generation

In [53]:
COUNT_FM_LAM0 = 1.0
COUNT_FM_NSTEP = 500
COUNT_FM_BATCH_EVAL = 32

gen_countFM_always = False
gen_countFM_OT_always = True

CFGEN_NSTEP = 50
CFGEN_BATCH = 512
SCLDM_NSTEP = 100
SCLDM_BATCH = 128
DCM_NSTEP = 100
DCM_TEMP  = 1.0

In [54]:
n_gen = X_test.shape[0] if (N_GEN is None) else int(N_GEN)
print("Generating n_gen =", n_gen)

Generating n_gen = 586


In [55]:
import os, numpy as np, random, torch

GEN_CACHE_DIR = "saved_genSamps/gen_cache_dg_v1"
os.makedirs(GEN_CACHE_DIR, exist_ok=True)

def _as_int32(X):
    X = np.asarray(X)
    if not np.issubdtype(X.dtype, np.integer):
        X = np.rint(X).astype(np.int64)
    return X.astype(np.int32, copy=False)

def _cache_file(task, model, n_gen, extra="", rep=None, seed=None):
    parts = [task, model, f"n{int(n_gen)}"]
    if rep is not None:
        parts.append(f"rep{int(rep)}")
    if seed is not None:
        parts.append(f"seed{int(seed)}")
    if extra:
        parts.append(extra)
    return os.path.join(GEN_CACHE_DIR, "__".join(parts) + ".npy")

def load_or_make(task, model, n_gen, extra, make_fn, rep=None, seed=None):
    p = _cache_file(task, model, n_gen, extra=extra, rep=rep, seed=seed)
    if os.path.exists(p):
        return np.load(p)
    X = _as_int32(make_fn())
    np.save(p, X)
    return X

def _set_all_seeds(s):
    random.seed(int(s))
    np.random.seed(int(s))
    torch.manual_seed(int(s))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(s))

def _call_with_seed(seed, fn):
    if seed is not None:
        _set_all_seeds(seed)
    return fn()

def sample_countfm_eval(model, model_name, n_gen, always=False, rep=None, seed=None):
    if getattr(model, "_needs_ot_training", False):
        raise RuntimeError(
            f"{model_name} is only initialized from the non-OT checkpoint. "
            f"Run OT continuation training and save it first."
        )

    extra = f"nstep{COUNT_FM_NSTEP}_bs{COUNT_FM_BATCH_EVAL}_x0poisson_lam{COUNT_FM_LAM0}"

    make_fn = lambda: _call_with_seed(
        seed,
        lambda: model.sample(
            n_samples=n_gen,
            n_step=COUNT_FM_NSTEP,
            batch_eval=COUNT_FM_BATCH_EVAL,
            x0_mode="poisson",
            lam0=COUNT_FM_LAM0,
        ),
    )

    if always:
        return _as_int32(make_fn())

    task = "rep" if rep is not None else "single"
    return load_or_make(
        task=task,
        model=model_name,
        n_gen=n_gen,
        extra=extra,
        make_fn=make_fn,
        rep=rep,
        seed=seed,
    )

In [56]:
Xhat_countfm = sample_countfm_eval(
    countfm_model,
    "count-FM",
    n_gen=n_gen,
    always=gen_countFM_always,
    seed=SEED,
)

Xhat_countfm_ot = sample_countfm_eval(
    countfm_ot_v2_model,
    # countfm_ot_model,
    "count-FM-OT",
    n_gen=n_gen,
    always=gen_countFM_OT_always,
    seed=SEED,
)

Xhat_scvi = load_or_make(
    task="single",
    model="scVI",
    n_gen=n_gen,
    extra="",
    make_fn=lambda: scvi_model.sample(n_samples=n_gen),
)

Xhat_scd = load_or_make(
    task="single",
    model="scDiffusion",
    n_gen=n_gen,
    extra="bs1000",
    make_fn=lambda: scdiff_model.sample(n_samples=n_gen, batch_size=1000),
)

Xhat_cfgen = load_or_make(
    task="single",
    model="CFGen",
    n_gen=n_gen,
    extra=f"nstep{CFGEN_NSTEP}_bs{CFGEN_BATCH}",
    make_fn=lambda: cfgen_model.sample(
        n_samples=n_gen, n_steps=CFGEN_NSTEP, batch_size=CFGEN_BATCH, unconditional=True
    ),
)

Xhat_scldm = load_or_make(
    task="single",
    model="scLDM",
    n_gen=n_gen,
    extra=f"nstep{SCLDM_NSTEP}_bs{SCLDM_BATCH}",
    make_fn=lambda: scldm_model.sample(
        n_samples=n_gen, n_steps=SCLDM_NSTEP, batch_size=SCLDM_BATCH, unconditional=True
    ),
)

Xhat_dcm = load_or_make(
    task="single",
    model="DCM",
    n_gen=n_gen,
    extra=f"nstep{DCM_NSTEP}_temp{DCM_TEMP}_chunk32",
    make_fn=lambda: dcm_model.sample(
        n_samples=n_gen, n_steps=DCM_NSTEP, temperature=DCM_TEMP, gen_chunk=max(1, min(32, n_gen))
    ),
)

In [57]:
def _cell_stats(Xc):
    lib = Xc.sum(axis=1)
    zfrac = (Xc == 0).mean(axis=1)
    return lib, zfrac

lib_r, z_r         = _cell_stats(X_test)
lib_cf, z_cf       = _cell_stats(Xhat_countfm)
lib_cf_ot, z_cf_ot = _cell_stats(Xhat_countfm_ot)
lib_sv, z_sv       = _cell_stats(Xhat_scvi)
lib_scd, z_scd     = _cell_stats(Xhat_scd)
lib_cg, z_cg       = _cell_stats(Xhat_cfgen)
lib_scldm, z_scldm = _cell_stats(Xhat_scldm)
lib_dcm, z_dcm     = _cell_stats(Xhat_dcm)

print("count-FM:", Xhat_countfm.shape, "min/max:", Xhat_countfm.min(), Xhat_countfm.max(),
      "| lib(mean):", lib_cf.mean(), "zero_frac(mean):", z_cf.mean())
print("count-FM-OT:", Xhat_countfm_ot.shape, "min/max:", Xhat_countfm_ot.min(), Xhat_countfm_ot.max(),
      "| lib(mean):", lib_cf_ot.mean(), "zero_frac(mean):", z_cf_ot.mean())
print("scVI:", Xhat_scvi.shape, "min/max:", Xhat_scvi.min(), Xhat_scvi.max(),
      "| lib(mean):", lib_sv.mean(), "zero_frac(mean):", z_sv.mean())
print("scDiffusion:", Xhat_scd.shape, "min/max:", Xhat_scd.min(), Xhat_scd.max(),
      "| lib(mean):", lib_scd.mean(), "zero_frac(mean):", z_scd.mean())
print("CFGen:", Xhat_cfgen.shape, "min/max:", Xhat_cfgen.min(), Xhat_cfgen.max(),
      "| lib(mean):", lib_cg.mean(), "zero_frac(mean):", z_cg.mean())
print("scLDM:", Xhat_scldm.shape, "min/max:", Xhat_scldm.min(), Xhat_scldm.max(),
      "| lib(mean):", lib_scldm.mean(), "zero_frac(mean):", z_scldm.mean())
print("DCM:", Xhat_dcm.shape, "min/max:", Xhat_dcm.min(), Xhat_dcm.max(),
      "| lib(mean):", lib_dcm.mean(), "zero_frac(mean):", z_dcm.mean())
print("real test:", X_test.shape, "min/max:", X_test.min(), X_test.max(),
      "| lib(mean):", lib_r.mean(), "zero_frac(mean):", z_r.mean())

count-FM: (586, 13913) min/max: 0 218 | lib(mean): 2122.4624573378837 zero_frac(mean): 0.9262395839184949
count-FM-OT: (586, 13913) min/max: 0 213 | lib(mean): 2408.032423208191 zero_frac(mean): 0.9209578587953565
scVI: (586, 13913) min/max: 0 724 | lib(mean): 2417.0546075085326 zero_frac(mean): 0.9090786994460212
scDiffusion: (586, 13913) min/max: 0 2321 | lib(mean): 2913.047781569966 zero_frac(mean): 0.910857795236071
CFGen: (586, 13913) min/max: 0 82831 | lib(mean): 3024.8890784982937 zero_frac(mean): 0.9303078933469789
scLDM: (586, 13913) min/max: 0 923 | lib(mean): 2886.4692832764504 zero_frac(mean): 0.8996639035998694
DCM: (586, 13913) min/max: 0 1281 | lib(mean): 2446.3139931740616 zero_frac(mean): 0.9019912626220129
real test: (586, 13913) min/max: 0 1718 | lib(mean): 2892.7713310580207 zero_frac(mean): 0.897968825777154


# 6. Evaluation

## PCA space

In [58]:
# ---- metrics (rebuild featurizer with principled PCA_DIM) ----

# choose PCA dim by explained variance on REAL test (after gene selection + normalization inside Featurizer)
# goal: keep smallest k such that cumvar >= VAR_TARGET, with bounds [PCA_MIN, PCA_MAX]
VAR_TARGET = 0.90
PCA_MIN = 20
PCA_MAX = 500
# G_KEEP = X_test.shape[1]
G_KEEP = 2000

# IMPORTANT: rebuild feat with pca_dim=None so we can inspect PCA spectrum on the selected genes
feat0 = Featurizer(
    target_sum=TARGET_SUM,
    g_keep=G_KEEP,
    pca_dim=None,
    seed=SEED,
)
feat0.fit(X_test)  # use the evaluation reference distribution
Z0 = feat0.transform(X_test)  # this should be (n, G_KEEP) if your Featurizer is written as intended

# fit PCA spectrum on Z0 to pick k
from sklearn.decomposition import PCA
pca_full = PCA(n_components=min(G_KEEP, Z0.shape[0]-1), random_state=SEED)
pca_full.fit(Z0)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.searchsorted(cumvar, VAR_TARGET) + 1)
k = max(PCA_MIN, min(PCA_MAX, k))

print(f"[eval] PCA_DIM selected = {k} (cumvar={cumvar[k-1]:.3f}, target={VAR_TARGET}, G_KEEP={G_KEEP})")

# now rebuild the actual featurizer used for metrics with this PCA_DIM
feat = Featurizer(
    target_sum=TARGET_SUM,
    g_keep=G_KEEP,
    pca_dim=k,
    seed=SEED,
)
feat.fit(X_test)

Z_real  = feat.transform(X_test)
Z_cf    = feat.transform(Xhat_countfm)
Z_cf_ot = feat.transform(Xhat_countfm_ot)
Z_sv    = feat.transform(Xhat_scvi)
Z_scd   = feat.transform(Xhat_scd)
Z_cg    = feat.transform(Xhat_cfgen)
Z_scldm = feat.transform(Xhat_scldm)
Z_dcm   = feat.transform(Xhat_dcm)

sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
sigma2_fixed = max(sigma2_fixed, 1e-6)

w2_cf    = w2_gaussian(Z_real, Z_cf)
w2_cf_ot = w2_gaussian(Z_real, Z_cf_ot)
w2_sv    = w2_gaussian(Z_real, Z_sv)
w2_scd   = w2_gaussian(Z_real, Z_scd)
w2_cg    = w2_gaussian(Z_real, Z_cg)
w2_scldm = w2_gaussian(Z_real, Z_scldm)
w2_dcm   = w2_gaussian(Z_real, Z_dcm)

mmd_cf, _    = mmd2_rbf_unbiased(Z_real, Z_cf,    sigma2=sigma2_fixed)
mmd_cf_ot, _ = mmd2_rbf_unbiased(Z_real, Z_cf_ot, sigma2=sigma2_fixed)
mmd_sv, _    = mmd2_rbf_unbiased(Z_real, Z_sv,    sigma2=sigma2_fixed)
mmd_scd, _   = mmd2_rbf_unbiased(Z_real, Z_scd,   sigma2=sigma2_fixed)
mmd_cg, _    = mmd2_rbf_unbiased(Z_real, Z_cg,    sigma2=sigma2_fixed)
mmd_scldm, _ = mmd2_rbf_unbiased(Z_real, Z_scldm, sigma2=sigma2_fixed)
mmd_dcm, _   = mmd2_rbf_unbiased(Z_real, Z_dcm,   sigma2=sigma2_fixed)

res = pd.DataFrame([
    {"model": "count-FM",    "W2": w2_cf,    "MMD2_RBF": mmd_cf},
    {"model": "count-FM-OT", "W2": w2_cf_ot, "MMD2_RBF": mmd_cf_ot},
    {"model": "scVI",        "W2": w2_sv,    "MMD2_RBF": mmd_sv},
    {"model": "scDiffusion", "W2": w2_scd,   "MMD2_RBF": mmd_scd},
    {"model": "CFGen",       "W2": w2_cg,    "MMD2_RBF": mmd_cg},
    {"model": "scLDM",       "W2": w2_scldm, "MMD2_RBF": mmd_scldm},
    {"model": "DCM",         "W2": w2_dcm,   "MMD2_RBF": mmd_dcm},
]).sort_values("MMD2_RBF")

res

[eval] PCA_DIM selected = 403 (cumvar=0.900, target=0.9, G_KEEP=2000)


,model,W2,MMD2_RBF
1,count-FM-OT,20.686548,0.018089
0,count-FM,20.462810,0.018360
5,scLDM,20.730997,0.020330
3,scDiffusion,20.604435,0.022320
2,scVI,20.633632,0.022964
6,DCM,23.556576,0.029586
4,CFGen,22.110139,0.035171


In [59]:
# import matplotlib.pyplot as plt

# if Z_real.shape[1] >= 2:
#     plt.figure(figsize=(6,6))
#     plt.scatter(Z_real[:,0],  Z_real[:,1],  s=6, alpha=0.25, label="real test")
#     plt.scatter(Z_cf[:,0],    Z_cf[:,1],    s=6, alpha=0.25, label="count-FM")
#     plt.scatter(Z_sv[:,0],    Z_sv[:,1],    s=6, alpha=0.25, label="scVI")
#     plt.scatter(Z_scd[:,0],   Z_scd[:,1],   s=6, alpha=0.25, label="scDiffusion")
#     plt.scatter(Z_cg[:,0],    Z_cg[:,1],    s=6, alpha=0.25, label="CFGen")
#     plt.scatter(Z_scldm[:,0], Z_scldm[:,1], s=6, alpha=0.25, label="scLDM")
#     plt.scatter(Z_dcm[:,0],   Z_dcm[:,1],   s=6, alpha=0.25, label="DCM")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

## multiple generations

In [60]:
# ===== Replication eval (uses current feat) =====
import time, numpy as np, pandas as pd

N_REP = 5
GEN_SEEDS = [SEED + i for i in range(N_REP)]
model_names = ["count-FM", "count-FM-OT", "scVI", "scDiffusion", "CFGen", "scLDM", "DCM"]

Z_real = feat.transform(X_test)
sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
sigma2_fixed = max(sigma2_fixed, 1e-6)
print("feat_dim =", Z_real.shape[1], "| sigma2 =", sigma2_fixed)

w2_runs  = {m: [] for m in model_names}
mmd_runs = {m: [] for m in model_names}

def _scldm_sample_rep(n_samples, n_steps, batch_size, rep_id, unconditional=True):
    n = int(n_samples)
    chunks, got = [], 0
    gen_idx = 1000 * int(rep_id)
    while got < n:
        Xg = scldm_model._infer_one_round(
            gen_idx=gen_idx,
            timesteps=int(n_steps),
            test_batch_size=int(batch_size),
            unconditional=bool(unconditional),
        )
        chunks.append(Xg)
        got += Xg.shape[0]
        gen_idx += 1
    return np.concatenate(chunks, axis=0)[:n]

t0 = time.time()
for r, s in enumerate(GEN_SEEDS):
    print(f"\nrep {r+1}/{N_REP} seed={s}")
    _set_all_seeds(s)

    if hasattr(scdiff_model, "seed"):
        scdiff_model.seed = int(s)
    if hasattr(dcm_model, "seed"):
        dcm_model.seed = int(s)
    if hasattr(scldm_model, "seed"):
        scldm_model.seed = int(s)

    X_cf = sample_countfm_eval(
        countfm_model,
        "count-FM",
        n_gen=n_gen,
        always=gen_countFM_always,
        rep=r,
        seed=s,
    )

    X_cf_ot = sample_countfm_eval(
        countfm_ot_v2_model,
        # countfm_ot_model,
        "count-FM-OT",
        n_gen=n_gen,
        always=gen_countFM_OT_always,
        rep=r,
        seed=s,
    )

    X_sv = load_or_make(
        task="rep",
        model="scVI",
        n_gen=n_gen,
        rep=r,
        seed=s,
        extra="",
        make_fn=lambda: scvi_model.sample(n_samples=n_gen),
    )

    X_scd = load_or_make(
        task="rep",
        model="scDiffusion",
        n_gen=n_gen,
        rep=r,
        seed=s,
        extra="bs1000",
        make_fn=lambda: scdiff_model.sample(n_samples=n_gen, batch_size=1000),
    )

    X_cg = load_or_make(
        task="rep",
        model="CFGen",
        n_gen=n_gen,
        rep=r,
        seed=s,
        extra=f"nstep{CFGEN_NSTEP}_bs{CFGEN_BATCH}",
        make_fn=lambda: cfgen_model.sample(
            n_samples=n_gen, n_steps=CFGEN_NSTEP, batch_size=CFGEN_BATCH, unconditional=True
        ),
    )

    X_scldm = load_or_make(
        task="rep",
        model="scLDM",
        n_gen=n_gen,
        rep=r,
        seed=s,
        extra=f"nstep{SCLDM_NSTEP}_bs{SCLDM_BATCH}_genidx{1000*r}",
        make_fn=lambda: _scldm_sample_rep(
            n_gen, SCLDM_NSTEP, SCLDM_BATCH, rep_id=r, unconditional=True
        ),
    )

    X_dcm = load_or_make(
        task="rep",
        model="DCM",
        n_gen=n_gen,
        rep=r,
        seed=s,
        extra=f"nstep{DCM_NSTEP}_temp{DCM_TEMP}_chunk32",
        make_fn=lambda: dcm_model.sample(
            n_samples=n_gen, n_steps=DCM_NSTEP, temperature=DCM_TEMP, gen_chunk=max(1, min(32, n_gen))
        ),
    )

    Z_cf    = feat.transform(X_cf)
    Z_cf_ot = feat.transform(X_cf_ot)
    Z_sv    = feat.transform(X_sv)
    Z_scd   = feat.transform(X_scd)
    Z_cg    = feat.transform(X_cg)
    Z_scldm = feat.transform(X_scldm)
    Z_dcm   = feat.transform(X_dcm)

    w2_runs["count-FM"].append(float(w2_gaussian(Z_real, Z_cf)))
    w2_runs["count-FM-OT"].append(float(w2_gaussian(Z_real, Z_cf_ot)))
    w2_runs["scVI"].append(float(w2_gaussian(Z_real, Z_sv)))
    w2_runs["scDiffusion"].append(float(w2_gaussian(Z_real, Z_scd)))
    w2_runs["CFGen"].append(float(w2_gaussian(Z_real, Z_cg)))
    w2_runs["scLDM"].append(float(w2_gaussian(Z_real, Z_scldm)))
    w2_runs["DCM"].append(float(w2_gaussian(Z_real, Z_dcm)))

    mmd_runs["count-FM"].append(float(mmd2_rbf_unbiased(Z_real, Z_cf, sigma2=sigma2_fixed)[0]))
    mmd_runs["count-FM-OT"].append(float(mmd2_rbf_unbiased(Z_real, Z_cf_ot, sigma2=sigma2_fixed)[0]))
    mmd_runs["scVI"].append(float(mmd2_rbf_unbiased(Z_real, Z_sv, sigma2=sigma2_fixed)[0]))
    mmd_runs["scDiffusion"].append(float(mmd2_rbf_unbiased(Z_real, Z_scd, sigma2=sigma2_fixed)[0]))
    mmd_runs["CFGen"].append(float(mmd2_rbf_unbiased(Z_real, Z_cg, sigma2=sigma2_fixed)[0]))
    mmd_runs["scLDM"].append(float(mmd2_rbf_unbiased(Z_real, Z_scldm, sigma2=sigma2_fixed)[0]))
    mmd_runs["DCM"].append(float(mmd2_rbf_unbiased(Z_real, Z_dcm, sigma2=sigma2_fixed)[0]))

print("\nDone in", round(time.time() - t0, 1), "sec")

def _mean_std(xs):
    xs = np.asarray(xs, dtype=float)
    return float(xs.mean()), float(xs.std(ddof=1)) if xs.size > 1 else 0.0

rows = []
for m in model_names:
    w_mu, w_sd = _mean_std(w2_runs[m])
    k_mu, k_sd = _mean_std(mmd_runs[m])
    rows.append({
        "model": m,
        "W2": f"{w_mu:.3f} ± {w_sd:.3f}",
        "MMD2_RBF": f"{k_mu:.4f} ± {k_sd:.4f}",
        "W2_mean": w_mu, "W2_std": w_sd,
        "MMD2_mean": k_mu, "MMD2_std": k_sd,
    })

res_rep = pd.DataFrame(rows).sort_values("MMD2_mean")
res_rep[["model","W2","MMD2_RBF","W2_mean","W2_std","MMD2_mean","MMD2_std"]]

feat_dim = 403 | sigma2 = 2676.56787109375

rep 1/5 seed=0

rep 2/5 seed=1

rep 3/5 seed=2

rep 4/5 seed=3

rep 5/5 seed=4

Done in 148.1 sec


,model,W2,MMD2_RBF,W2_mean,W2_std,MMD2_mean,MMD2_std
1,count-FM-OT,20.592 ± 0.058,0.0179 ± 0.0005,20.592031,0.058435,0.017885,0.000483
0,count-FM,20.456 ± 0.055,0.0185 ± 0.0005,20.456392,0.054851,0.018478,0.000510
5,scLDM,20.739 ± 0.018,0.0202 ± 0.0003,20.738714,0.017587,0.020225,0.000287
3,scDiffusion,20.545 ± 0.051,0.0225 ± 0.0004,20.544957,0.050611,0.022490,0.000443
2,scVI,20.687 ± 0.042,0.0229 ± 0.0002,20.686727,0.041688,0.022865,0.000166
6,DCM,23.566 ± 0.010,0.0296 ± 0.0000,23.565870,0.010270,0.029584,0.000027
4,CFGen,22.226 ± 0.106,0.0361 ± 0.0008,22.226417,0.105776,0.036139,0.000791


In [61]:
res_rep

,model,W2,MMD2_RBF,W2_mean,W2_std,MMD2_mean,MMD2_std
1,count-FM-OT,20.592 ± 0.058,0.0179 ± 0.0005,20.592031,0.058435,0.017885,0.000483
0,count-FM,20.456 ± 0.055,0.0185 ± 0.0005,20.456392,0.054851,0.018478,0.000510
5,scLDM,20.739 ± 0.018,0.0202 ± 0.0003,20.738714,0.017587,0.020225,0.000287
3,scDiffusion,20.545 ± 0.051,0.0225 ± 0.0004,20.544957,0.050611,0.022490,0.000443
2,scVI,20.687 ± 0.042,0.0229 ± 0.0002,20.686727,0.041688,0.022865,0.000166
6,DCM,23.566 ± 0.010,0.0296 ± 0.0000,23.565870,0.010270,0.029584,0.000027
4,CFGen,22.226 ± 0.106,0.0361 ± 0.0008,22.226417,0.105776,0.036139,0.000791


In [62]:
# # res_rep
# res_rep_old = res_rep.copy()

In [63]:
# # ===== recompute ONLY count-FM row, keep others from res_rep_old =====
# import numpy as np, time, random, torch, pandas as pd

# # must match the previous replication settings
# N_REP = 5
# GEN_SEEDS = [SEED + i for i in range(N_REP)]

# # these must be the same as used to build res_rep_old
# Z_real = feat.transform(X_test)
# sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
# sigma2_fixed = max(sigma2_fixed, 1e-6)

# def _set_all_seeds(s):
#     random.seed(int(s))
#     np.random.seed(int(s))
#     torch.manual_seed(int(s))
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(int(s))

# w2_list, mmd_list = [], []
# t0 = time.time()
# for r, s in enumerate(GEN_SEEDS):
#     _set_all_seeds(s)

#     X_cf = countfm_model.sample(
#         n_samples=n_gen,
#         n_step=COUNT_FM_NSTEP,
#         batch_eval=COUNT_FM_BATCH_EVAL,
#         x0_mode="poisson",
#         lam0=COUNT_FM_LAM0,
#     )
#     Z_cf = feat.transform(X_cf)

#     w2_list.append(float(w2_gaussian(Z_real, Z_cf)))
#     mmd_list.append(float(mmd2_rbf_unbiased(Z_real, Z_cf, sigma2=sigma2_fixed)[0]))

# print("count-FM recompute time:", round(time.time()-t0, 1), "sec")

# w2_mu = float(np.mean(w2_list))
# w2_sd = float(np.std(w2_list, ddof=1)) if len(w2_list) > 1 else 0.0
# mmd_mu = float(np.mean(mmd_list))
# mmd_sd = float(np.std(mmd_list, ddof=1)) if len(mmd_list) > 1 else 0.0

# new_cf_row = {
#     "model": "count-FM",
#     "W2": f"{w2_mu:.3f} ± {w2_sd:.3f}",
#     "MMD2_RBF": f"{mmd_mu:.4f} ± {mmd_sd:.4f}",
#     "W2_mean": w2_mu, "W2_std": w2_sd,
#     "MMD2_mean": mmd_mu, "MMD2_std": mmd_sd,
# }

# # splice into old table
# res_rep_new = res_rep_old.copy()
# res_rep_new = res_rep_new[res_rep_new["model"] != "count-FM"]
# res_rep_new = pd.concat([res_rep_new, pd.DataFrame([new_cf_row])], ignore_index=True)
# res_rep_new = res_rep_new.sort_values("MMD2_mean")

# res_rep_new[["model","W2","MMD2_RBF","W2_mean","W2_std","MMD2_mean","MMD2_std"]]